# Unidade 1 - Bloco prático da Aula 01: a curva em U

Ajusta um modelo polinomial de grau 12 com regularização Ridge, varrendo o hiperparâmetro `alpha` em 30 pontos numa escala logarítmica, e mede o erro separadamente no treino e na validação. Semente fixa (`seed=42`) para reprodutibilidade.

Rode a célula de código e depois a célula de gráfico para reproduzir a Figura 4 da apostila.

In [ ]:
import numpy as np
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

rng = np.random.default_rng(seed=42)

# Dados sinteticos: padrao suave (senoide) + ruido conhecido
X = rng.uniform(0, 1, size=(80, 1))
y = np.sin(2 * np.pi * X).ravel() + rng.normal(0, 0.25, size=80)

X_tr, X_val, y_tr, y_val = train_test_split(
    X, y, test_size=0.4, random_state=42
)

# Varredura do hiperparametro de regularizacao
alphas = np.logspace(-8, 2, 30)
erros_treino, erros_val = [], []

for alpha in alphas:
    modelo = make_pipeline(
        PolynomialFeatures(degree=12),   # flexivel o bastante p/ decorar
        StandardScaler(),
        Ridge(alpha=alpha)               # alpha: preco da complexidade
    )
    modelo.fit(X_tr, y_tr)
    erros_treino.append(mean_squared_error(y_tr, modelo.predict(X_tr)))
    erros_val.append(mean_squared_error(y_val, modelo.predict(X_val)))

melhor = alphas[int(np.argmin(erros_val))]
print(f"Melhor alpha na validacao: {melhor:.2e}")
print(f"Erro de treino nesse alpha:    {erros_treino[int(np.argmin(erros_val))]:.4f}")
print(f"Erro de validacao nesse alpha: {min(erros_val):.4f}")

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4.5))
plt.plot(alphas, erros_treino, marker="o", label="erro no treinamento")
plt.plot(alphas, erros_val, marker="s", label="erro na validacao")
plt.axvline(melhor, color="green", linestyle=":", label=f"melhor equilibrio (alpha={melhor:.0e})")
plt.xscale("log")
plt.gca().invert_xaxis()
plt.xlabel("regularizacao menor  ->  modelo mais complexo")
plt.ylabel("erro quadratico medio")
plt.legend()
plt.title("Curva em U: o treino sempre melhora, a validacao conta a verdade")
plt.tight_layout()
plt.show()